# Disease Association Network Analysis
**Data**: `/api/v1/export/disease-network`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import requests
import warnings
warnings.filterwarnings('ignore')
API_BASE_URL = 'http://localhost:8000/api/v1'
print('Setup complete')

In [ ]:
diseases = ['diabetes', 'alzheimer', 'cancer', 'cardiovascular', 'autism']
networks = {}
for d in diseases:
    r = requests.get(f'{API_BASE_URL}/export/disease-network', params={'trait_name': d, 'limit': 5000})
    if r.status_code == 200:
        networks[d] = r.json()
        print(f'{d}: {len(r.json()["nodes"])} nodes, {len(r.json()["edges"])} edges')

## Diabetes Network Analysis

In [ ]:
data = networks.get('diabetes', {})
if data:
    G = nx.Graph()
    for n in data['nodes']: G.add_node(n['id'], name=n['name'], node_type=n['type'])
    for e in data['edges']: G.add_edge(e['source'], e['target'])
    print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
    print(f'lncRNA: {sum(1 for n,d in G.nodes(data=True) if d["node_type"]=="lncrna")}')

In [ ]:
degree = nx.degree_centrality(G)
between = nx.betweenness_centrality(G)
lncrna_cent = [{'id': n, 'name': d['name'], 'degree': degree[n], 'betweenness': between[n]}
    for n,d in G.nodes(data=True) if d['node_type']=='lncrna']
ldf = pd.DataFrame(lncrna_cent).sort_values('betweenness', ascending=False)
print('Top 15 lncRNA targets:')
print(ldf.head(15))
ldf.to_excel('results/diabetes_lncrna_targets.xlsx', index=False)

In [ ]:
ldf['score'] = ldf['degree'] * ldf['betweenness']
top = ldf.nlargest(20, 'score')
plt.figure(figsize=(12, 8))
plt.barh(top['name'], top['score'], color=sns.color_palette('rocket_r', 20))
plt.xlabel('Composite Score')
plt.title('Top 20 Therapeutic Targets for Diabetes', fontweight='bold')
plt.gca().invert_yaxis()
plt.savefig('figures/14_diabetes_therapeutic_targets.png', dpi=300)
plt.show()

## Multi-Disease Overlap

In [ ]:
disease_lnc = {d: set(n['name'] for n in data['nodes'] if n['type']=='lncrna') for d,data in networks.items()}
for d,lncs in disease_lnc.items(): print(f'{d}: {len(lncs)} lncRNAs')
names = list(disease_lnc.keys())
matrix = np.zeros((len(names), len(names)), dtype=int)
for i,d1 in enumerate(names):
    for j,d2 in enumerate(names):
        if i==j: matrix[i,j] = len(disease_lnc[d1])
        else: matrix[i,j] = len(disease_lnc[d1] & disease_lnc[d2])
shared = pd.DataFrame(matrix, index=names, columns=names)
print(shared)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(shared, annot=True, fmt='d', cmap='Blues', square=True)
plt.title('Shared lncRNAs Between Diseases', fontweight='bold')
plt.savefig('figures/15_disease_shared_lncrnas.png', dpi=300)
plt.show()

In [ ]:
print('KEY FINDINGS')
for d,data in networks.items():
    print(f'{d}: {len(data["nodes"])} nodes')